#  **Unstructured - PDF RAG** 



---

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from glob import glob

from pprint import pprint
import json

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [3]:
import logging

# 로깅 레벨 설정 
logging.getLogger('pdfminer').setLevel(logging.ERROR)
logging.getLogger('unstructured').setLevel(logging.ERROR)

---

## **테슬라 10-K 리포트 PDF 기반 RAG 시스템 구축 (Unstructured & Langchain)**

- **테슬라 10-K 보고서** PDF를 `unstructured` 라이브러리로 파싱

- **Langchain**과 **ChromaDB**를 활용한 RAG 시스템 구축

- PDF 문서 기반 **지능형 질의응답** 시스템 구현 프로젝트

### 1. **문서 로드**

- **테슬라 10-K 보고서** PDF 파일을 **인터넷에서 다운로드**

- **SEC EDGAR 웹사이트** 또는 **테슬라 IR 페이지**에서 다운로드 가능

- 출처: https://ir.tesla.com/#quarterly-disclosure

In [4]:
from langchain_community.document_loaders import PyPDFLoader

# PDF 파일 경로 지정
file_path = "data/tsla-20241231-gen.pdf"

# PyPDFLoader 초기화
loader = PyPDFLoader(file_path)

# PDF 문서 로드
pypdf_docs = loader.load()

# 로드된 문서 개수 확인
print(len(pypdf_docs)) 

W1115 11:50:53.120000 29156 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


144


In [5]:
# 첫 번째 페이지의 내용 출력
print(f"{pypdf_docs[0].page_content}\n")
print("-" * 100)

# 페이지 메타데이터 확인 
pprint(pypdf_docs[0].metadata)

UNITED	STATES
SECURITIES	AND	EXCHANGE	COMMISSION
Washington,	D.C.	20549
FORM	
10-K
(Mark	One)
x
ANNUAL	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	fiscal	year	ended	
December	31
,	2024
OR
o
TRANSITION	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	transition	period	from	_________	to	_________
Commission	File	Number:	
001-34756
Tesla,	Inc.
(Exact	name	of	registrant	as	specified	in	its	charter)
Texas
91-2197729
(State	or	other	jurisdiction	of
incorporation	or	organization)
(I.R.S.	Employer
Identification	No.)
1	Tesla	Road
Austin
,	
Texas
78725
(Address	of	principal	executive	offices)
(Zip	Code)
(
512
)	
516-8177
(Registrant’s	telephone	number,	including	area	code)
Securities	registered	pursuant	to	Section	12(b)	of	the	Act:
Title	of	each	class
Trading	Symbol(s)
Name	of	each	exchange	on	which	registered
Common	stock
TSLA
The	Nasdaq	Global	Select	Market
Securities	registered	pursuant	to	Section	12(g)	of	the	Act

### 2. **문서 파싱 (Unstructured)**

- **Unstructured 라이브러리**를 사용하여 **PDF 문서 파싱** 및 텍스트 추출을 수행

`(1) 문서 로드`

- LangChain에서 제공하는 UnstructuredLoader 사용   

In [6]:
# 추출한 이미지 저장 폴더 
image_folder = "data/images/tesla_10k"
os.makedirs(image_folder, exist_ok=True)

In [7]:
# docs = pickled_docs

In [ ]:
# ✅ import는 이미 올바르게 되어 있음
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_community.document_loaders import UnstructuredFileLoader

from unstructured.cleaners.core import (
    clean_extra_whitespace,
    replace_unicode_quotes,
    clean_non_ascii_chars,
    group_broken_paragraphs
)

# ✅ 로더 생성 - 이 한 줄만 수정
loader = UnstructuredFileLoader(  # UnstructuredLoader → UnstructuredFileLoader
    # PDF 파일 경로
    file_path,             

    # 파티셔닝 전략 설정
    strategy="hi_res",                  
    hi_res_model_name="yolox",          
    infer_table_structure=True,         
    languages=["eng"],           
    
    # 이미지 추출 설정
    extract_images_in_pdf=True,         
    extract_image_block_types=["Image", "Table"],
    extract_image_block_output_dir=image_folder,  

    # 후처리 설정
    post_processors=[
        clean_extra_whitespace,  # 불필요한 공백 제거
        replace_unicode_quotes,  # 유니코드 따옴표 제거
        clean_non_ascii_chars,   # 비 ASCII 문자 제거
        group_broken_paragraphs,  # 줄바꿈으로 분리된 문단 결합
    ], 
)

# ✅ 나머지 코드는 그대로 유지
docs = []
for doc in loader.lazy_load():
    docs.append(doc)

# 문서 개수 확인
print(len(docs))


C:\Users\kaydash\AppData\Local\Temp\ipykernel_29156\1443924642.py:13: LangChainDeprecationWarning: The class `UnstructuredFileLoader` was deprecated in LangChain 0.2.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-unstructured package and should be used instead. To use it run `pip install -U `langchain-unstructured` and import as `from `langchain_unstructured import UnstructuredLoader``.
  loader = UnstructuredFileLoader(  # UnstructuredLoader → UnstructuredFileLoader


In [ ]:
# 문서 출력
for doc in docs[:5]:
    pprint(doc.page_content)
    print("-" * 100)
    pprint(doc.metadata)
    print("=" * 100)
    print()

In [ ]:
# 첫 번째 문서의 내용 출력
pprint(docs[0].page_content)

In [ ]:
# 메타데이터 확인
pprint(docs[0].metadata)

In [ ]:
# 문서 객체를 pickle로 저장
import pickle
with open("data/tesla_10k.pkl", "wb") as f:
    pickle.dump(docs, f)

In [ ]:
from langchain_core.documents import Document

# pickle로 저장된 문서 객체 로드
with open("data/tesla_10k.pkl", "rb") as f:
    pickled_docs = pickle.load(f)

# 문서 개수 확인
print(len(pickled_docs))

# 첫 번째 문서의 내용 출력
print(f"{pickled_docs[0].page_content}\n")

print("-" * 100)

# 첫 번째 문서의 메타데이터 확인
pprint(pickled_docs[0].metadata)

`(2) 표 데이터 변환`

- 테이블 형식의 데이터를 마크다운 텍스트로 변환하여 처리 

In [ ]:
# 문서 구성 요소의 유형 확인
category_counts = {}

for doc in docs:
    category = doc.metadata["category"]
    if category in category_counts:
        category_counts[category] += 1
    else:
        category_counts[category] = 1

# 카테고리별 문서 개수 출력
pprint(category_counts)

In [ ]:
# 카테고리 별로 문서 재정리 

new_docs = []
for doc in docs:

    # Table 카테고리의 문서를 마크다운 형식으로 변환
    if doc.metadata["category"] == "Table":
        # 판다스 데이터프레임으로 변환
        _df = pd.read_html(doc.metadata['text_as_html'])[0]

        # 마크다운 형식으로 변환
        _md = _df.to_markdown(index=False)

        # 새로운 문서 객체 생성
        new_docs.append(Document(page_content=_md, metadata=doc.metadata))

    # Header, Footer, Image 카테고리의 문서는 제외
    elif doc.metadata["category"] in ["Header", "Footer", "Image"]:
        continue

    # 나머지 문서는 그대로 추가
    else:
        new_docs.append(doc)
    
# 변환된 문서 개수 확인
print(len(new_docs))

### 3. **텍스트 청킹 (Langchain)**

- **텍스트 청킹**은 대규모 텍스트를 의미 있는 **작은 단위로 분할**하는 기술

- 주요 목적은 **벡터 데이터베이스 저장 및 검색 효율성 향상**

`(1) 적정 청크 크기`

In [ ]:
# 각 문서의 page_content 길이 확인

doc_lengths = [len(doc.page_content) for doc in new_docs]

# 문서 길이 시각화
import matplotlib.pyplot as plt

plt.hist(doc_lengths, bins=50)
plt.show()

In [ ]:
# 문서 타입별 개수 확인
category_counts = {}

for doc in new_docs:
    category = doc.metadata["category"]
    if category in category_counts:
        category_counts[category] += 1
    else:
        category_counts[category] = 1

# 카테고리별 문서 개수 출력
pprint(category_counts)

In [ ]:
# 문서 타입별로 문서 길이 확인
category_lengths = {}
for doc in new_docs:
    category = doc.metadata["category"]
    if category not in category_lengths:
        category_lengths[category] = []
    category_lengths[category].append(len(doc.page_content))

# 카테고리별 문서 길이 시각화
for category, lengths in category_lengths.items():
    plt.hist(lengths, bins=50)
    plt.title(f"Document Lengths for {category}")
    plt.xlabel("Length")
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
# 문서 타입별 토큰 수 확인 (tiktoken tokenizer 사용)

import tiktoken

# tiktoken tokenizer 초기화
tokenizer = tiktoken.get_encoding("cl100k_base")

# 문서 타입별로 토큰 수 확인
category_tokens = {}
for doc in new_docs:
    category = doc.metadata["category"]
    if category not in category_tokens:
        category_tokens[category] = []
    category_tokens[category].append(len(tokenizer.encode(doc.page_content)))
    
# 카테고리별 토큰 수 시각화
for category, tokens in category_tokens.items():
    plt.hist(tokens, bins=50)
    plt.title(f"Document Tokens for {category}")
    plt.xlabel("Tokens")
    plt.ylabel("Frequency")
    plt.show()

`(2) 목차 항목 추출`

In [ ]:
# 3페이지(page_number) 목차 항목을 기준으로 그룹화 

toc_items = [ doc.page_content for doc in new_docs if doc.metadata["page_number"] == 3]
toc_items

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

# 목차 항목 모델 정의
class Item(BaseModel):
    number: str = Field(description="목차 항목 번호 (예: 1, 1A, 1B, 2, 3 등)")
    title: str = Field(description="목차 항목 제목")

class Section(BaseModel):
    section: str = Field(description="목차 항목 그룹")
    items: list[Item] = Field(description="목차 항목 리스트")

# 목차 텍스트에서 항목을 추출하기 위한 프롬프트
TOC_PROMPT = """
You are a helpful assistant.
The user will provide you with a list of items.

Your task is to group these items into sections based on their content.

Please provide the output in JSON format.
The JSON should contain the following fields:
- section: The name of the section
- items: A list of items that belong to this section

The items are as follows:
{items}
"""

# 프롬프트 템플릿 생성
toc_prompt_template = PromptTemplate(
    input_variables=["items"],
    template=TOC_PROMPT
)

# LLM 정의 
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
llm_structured = llm.with_structured_output(Section)

# LCEL 체인 
toc_chain = toc_prompt_template | llm_structured

# 목차 항목을 기준으로 그룹화
toc_section = toc_chain.invoke({"items": "\n\n".join(toc_items)})

# 그룹화된 목차 항목 확인
pprint(toc_section)


In [ ]:
for item in toc_section.items:
    print(f"{item.number}: {item.title}")

In [ ]:
def get_item_header(item_number, item_title):
    """가능한 목차 항목 제목 변형을 반환하는 함수"""
    variations = [
        f"ITEM {item_number.upper()}. {item_title.upper()}",
    ]
    return variations

toc_based_docs = []
current_section = None

for i, doc in enumerate(new_docs):
    # 이 문서가 목차 항목에 해당하는지 확인
    new_section_found = False
    
    for item in toc_section.items:
        item_headers = get_item_header(item.number, item.title)
        
        # 목차 항목 제목 변형이 문서에 포함되어 있는지 확인
        if any(header in doc.page_content for header in item_headers):
            current_section = item.title
            new_section_found = True
            break

    # page_content에서 목차 항목 부분부터 추출
    if new_section_found:
        # 문서의 page_content에서 목차 항목 부분부터 추출
        start_index = min([doc.page_content.index(header) for header in item_headers if header in doc.page_content])
        doc.page_content = doc.page_content[start_index:]
    
    # 문서에 섹션 메타데이터 추가
    doc_copy = Document(
        page_content=doc.page_content,
        metadata={
            **doc.metadata,
            "section": current_section if current_section else "Unknown"
        }
    )
    toc_based_docs.append(doc_copy)

# 섹션별 문서 개수 확인
section_counts = {}
for doc in toc_based_docs:
    section = doc.metadata.get("section", "Unknown")
    section_counts[section] = section_counts.get(section, 0) + 1

print("Section distribution:")
for section, count in section_counts.items():
    print(f"{section}: {count} documents")

In [ ]:
# 목차 항목의 순서에 따라 섹션 정렬
section_order = {item.title: idx for idx, item in enumerate(toc_section.items)}

# "Unknown" 섹션을 마지막에 추가
section_order["Unknown"] = len(section_order)

# 섹션 순서에 따라 문서 정렬
toc_based_docs_sorted = sorted(
    toc_based_docs, 
    key=lambda x: section_order.get(x.metadata.get("section", "Unknown"), float('inf'))
)

# 섹션별 문서 개수 확인 (정렬된 문서)
section_counts_sorted = {}
for doc in toc_based_docs_sorted:
    section = doc.metadata.get("section", "Unknown")
    section_counts_sorted[section] = section_counts_sorted.get(section, 0) + 1

# 정렬된 섹션 분포 출력
print("\nSorted section distribution:")
for section in sorted(section_counts_sorted.keys(), key=lambda x: section_order.get(x, float('inf'))):
    print(f"{section}: {section_counts_sorted[section]} documents")

In [ ]:
# 문서 객체를 pickle로 저장
with open("data/tesla_10k_toc.pkl", "wb") as f:
    pickle.dump(toc_based_docs_sorted, f)

In [ ]:
# pickle로 저장된 문서 객체 로드
with open("data/tesla_10k_toc.pkl", "rb") as f:
    toc_based_docs_sorted = pickle.load(f)

# 문서 개수 확인
print(len(toc_based_docs_sorted))

`(3) 섹션별로 결합`

- 보고서 섹션별로 문서 객체를 결합하여 재구조화 

In [ ]:
# Unknown 제외 나머지 섹션별로 결합 

section_docs = {}

for doc in toc_based_docs_sorted:
    section = doc.metadata.get("section", "Unknown")
    if section == "Unknown":
        continue
    if section not in section_docs:
        section_docs[section] = []
    section_docs[section].append(doc)

# 섹션별로 문서 결합
section_docs_combined = {}
for section, docs in section_docs.items():
    combined_content = "\n\n".join([doc.page_content for doc in docs])
    combined_metadata = docs[0].metadata
    section_docs_combined[section] = Document(page_content=combined_content, metadata=combined_metadata)


# 결합된 문서 개수 확인
print(len(section_docs_combined))

In [ ]:
# 문서 저장
with open("data/tesla_10k_sections.pkl", "wb") as f:
    pickle.dump(section_docs_combined, f)

In [ ]:
# pickle로 저장된 문서 객체 로드
import pickle
with open("data/tesla_10k_sections.pkl", "rb") as f:
    section_docs_combined = pickle.load(f)

# 문서 개수 확인
print(len(section_docs_combined))

In [ ]:
# 첫 번째 문서의 내용 출력
print(f"{section_docs_combined['Business'].page_content}\n")

# 첫 번째 문서의 메타데이터 확인
pprint(section_docs_combined['Business'].metadata)

### 4. **Parent Document Retriever**

- 문서 검색 시 **작은 단위 분할**과 **문맥 유지** 사이의 균형이 중요함

- 작은 단위로 분할하면 **임베딩의 정확도**가 높아지나 문맥이 손실될 수 있음

- **ParentDocumentRetriever**는 작은 청크로 저장하고 검색 시 상위 문서를 반환하여 두 가지 목표를 달성함

- 효과적인 문서 검색을 위해 청크 크기와 문맥 보존 사이의 최적점을 찾는 것이 핵심

In [ ]:
# 각 섹션별로 문서 길이 확인
section_lengths = {section: len(doc.page_content) for section, doc in section_docs_combined.items()}

# 길이 순서로 정렬
section_lengths = dict(sorted(section_lengths.items(), key=lambda item: item[1]))

# 섹션별 문서 길이 시각화
plt.barh(section_lengths.keys(), section_lengths.values())
plt.xticks(rotation=90)
plt.xlabel("Section")
plt.ylabel("Length")
plt.title("Document Lengths by Section")
plt.show()

In [ ]:
import tiktoken

# tiktoken tokenizer 초기화
tokenizer = tiktoken.get_encoding("cl100k_base")

# 각 섹션별로 토큰 수 확인
section_tokens = {section: len(tokenizer.encode(doc.page_content)) for section, doc in section_docs_combined.items()}

# 길이 순서로 정렬
section_tokens = dict(sorted(section_tokens.items(), key=lambda item: item[1]))

# 섹션별 토큰 수 시각화
plt.barh(section_tokens.keys(), section_tokens.values())
plt.xticks(rotation=90)
plt.xlabel("Section")
plt.ylabel("Tokens")
plt.title("Document Tokens by Section")
plt.show()

In [ ]:
# 첫 번째 섹션의 메타데이터 확인
pprint(section_docs_combined['Business'].metadata)

In [ ]:
# 토큰 수 기준으로 3000개 이상이면 분할 (섹션 내의 순서 정보를 metadata에 추가)
from langchain_core.documents import Document
import tiktoken


tokenizer = tiktoken.get_encoding("cl100k_base")

section_docs_split = {}

for section, doc in section_docs_combined.items():

    filtered_metadata = {
        'element_id': doc.metadata['element_id'],
        'parent_id': doc.metadata['parent_id'] if 'parent_id' in doc.metadata else None,
        'source': doc.metadata['source'],
        'page_number': doc.metadata['page_number'],   
        'section': section        
    }

    tokens = len(tokenizer.encode(doc.page_content))
    if tokens > 3000:
        # 문서 분할
        split_docs = []
        for i in range(0, len(doc.page_content), 3000):
            split_doc = Document(
                page_content=doc.page_content[i:i+3000],
                metadata={
                    **filtered_metadata,
                    "order": i // 3000 + 1
                }
            )
            split_docs.append(split_doc)
        section_docs_split[section] = split_docs
    else:
        # 문서 그대로 추가
        doc.metadata = filtered_metadata
        doc.metadata["order"] = 1
        section_docs_split[section] = [doc]


# 분할된 문서 개수 확인
len([doc for docs in section_docs_split.values() for doc in docs])

In [ ]:
section_docs_split['Business'][0].metadata 

In [ ]:
# 문서 객체를 pickle로 저장
with open("data/tesla_10k_sections_split.pkl", "wb") as f:
    pickle.dump(section_docs_split, f)

In [ ]:
# pickle로 저장된 문서 객체 로드
with open("data/tesla_10k_sections_split.pkl", "rb") as f:
    section_docs_split = pickle.load(f)

# 문서 개수 확인
print(len([doc for docs in section_docs_split.values() for doc in docs]))

In [ ]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import LocalFileStore
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
import os
import pickle

# 문서 저장소 경로 설정
storage_path = "./document_store"
os.makedirs(storage_path, exist_ok=True)

# 부모 문서 저장소 클래스 정의
class PickleFileStore(LocalFileStore):
    def mget(self, keys):
        """Get the values for the given keys."""
        return [pickle.loads(v) if v is not None else None 
                for v in super().mget(keys)]

    def mset(self, key_value_pairs):
        """Set the values for the given key-value pairs."""
        serialized_pairs = [
            (k, pickle.dumps(v)) for k, v in key_value_pairs
        ]
        super().mset(serialized_pairs)

# 부모 문서 저장소 초기화
store = PickleFileStore(storage_path)

# 부모 문서용 텍스트 스플리터 (섹션 단위로 큰 청크)
parent_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=2000,
    chunk_overlap=400,
    separators=["\n\n", "\n", " ", ""]
)

# 자식 문서용 텍스트 스플리터 (검색용 작은 청크)
child_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

# 벡터 스토어 초기화
vectorstore = Chroma(
    collection_name="tesla_10k_sections",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory="./chroma_db"
)

# ParentDocumentRetriever 설정
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# 문서 추가
retriever.add_documents(
    [
        doc
        for docs in section_docs_split.values()
        for doc in docs
    ]
)

In [ ]:
# 저장된 문서 수 확인 
print(f"Total documents in vectorstore: {vectorstore._collection.count()}")

# 로컬 저장소 문서 수 확인
all_keys = list(store.yield_keys())
print(f"Total documents in store: {len(all_keys)}")


In [ ]:
# Test 검색
query = "What are the main risk factors?"
retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} documents")
for doc in retrieved_docs:
    print(doc.page_content[:500])
    print("-" * 100)

In [ ]:
# 벡터 스토어 로드
vectorstore = Chroma(
    collection_name="tesla_10k_sections",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory="./chroma_db"
)

# 로컬 파일 저장소 로드
storage_path = "./document_store"
store = PickleFileStore(storage_path)

# ParentDocumentRetriever 설정
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# Test 검색
query = "What are the main risk factors?"

retrieved_docs = retriever.invoke(query)
print(f"Retrieved {len(retrieved_docs)} documents")

for doc in retrieved_docs:
    print(doc.page_content[:500])
    print("-" * 100)

In [ ]:
# 다른 쿼리 

query = "Where is Tesla's headquarters located?"

retrieved_docs = retriever.invoke(query)
print(f"Retrieved {len(retrieved_docs)} documents")

for doc in retrieved_docs:
    print(doc.page_content[:500])
    print("-" * 100)

In [ ]:
# vectorstore.delete_collection()

### 5. **다중 벡터 기반 검색(Multi Vector Retrieval)**

- **Multi-Vector Retriever**는 문서를 **여러 벡터**로 분할하여 저장하고 검색하는 시스템
    1. 벡터 저장소(Vectorstore): 임베딩 저장
    2. 문서 저장소(Docstore): 원본 문서 저장

- 예제: **요약 기반 검색 시스템** 구현

    - 각 섹션의 간결한 요약문을 생성하여 벡터 저장소에 보관하고 유사도 검색에 활용함
    - **영구 저장소**는 PickleFileStore와 Chroma를 사용하여 문서와 벡터 데이터를 세션 간 유지함
    - **검색 과정**은 요약문으로 의미론적 매칭을 수행하고 전체 원본 문서를 반환하여 완전한 문맥을 제공함
    - **RAG 파이프라인**은 ChatGPT를 활용하여 요약 생성과 최종 답변을 처리하며 명확한 응답 형식을 제공함

In [ ]:
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import LocalFileStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import os
import pickle

# 로컬 파일 저장소 클래스 정의
class PickleFileStore(LocalFileStore):
    def mget(self, keys):
        return [pickle.loads(v) if v is not None else None 
                for v in super().mget(keys)]

    def mset(self, key_value_pairs):
        serialized_pairs = [(k, pickle.dumps(v)) for k, v in key_value_pairs]
        super().mset(serialized_pairs)

# 저장소 디렉토리 생성
os.makedirs("./summary_store", exist_ok=True)
os.makedirs("./chroma_db", exist_ok=True)

# 로컬 파일 저장소 초기화
doc_store = PickleFileStore("./summary_store")
vectorstore = Chroma(
    collection_name="tesla_10k_summaries",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory="./chroma_db"
)

# MultiVectorRetriever 설정
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=doc_store,
    id_key="doc_id"   # 문서 ID 키 설정
)

# 각 문서에 대한 요약 생성
def generate_summary(text):
    summary_prompt = ChatPromptTemplate.from_template(
        "Summarize the following 10-K report section in 2-3 sentences:\n\n{text}"
    )
    model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    summary_chain = summary_prompt | model | StrOutputParser()
    return summary_chain.invoke({"text": text})

# 요약된 문서 저장
summary_docs = []
original_docs = []
doc_ids = []

# 각 섹션별로 문서 결합
to_index_docs = [
        doc
        for docs in section_docs_split.values()
        for doc in docs
]

# 각 문서에 대한 요약 생성
for i, doc in enumerate(to_index_docs):
    doc_id = f"doc_{i}"
    summary = generate_summary(doc.page_content)
    
    # 요약 문서 생성
    summary_doc = Document(
        page_content=summary,
        metadata={"doc_id": doc_id, "section": doc.metadata["section"]}
    )
    summary_docs.append(summary_doc)
    
    # 원본 문서와 ID 저장
    original_docs.append(doc)
    doc_ids.append(doc_id)

# 벡터 스토어에 요약 문서 추가
retriever.vectorstore.add_documents(summary_docs)
retriever.docstore.mset(list(zip(doc_ids, original_docs)))

In [ ]:
# 로컬 파일 저장소 로드
doc_store = PickleFileStore("./summary_store")
vectorstore = Chroma(
    collection_name="tesla_10k_summaries",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory="./chroma_db"
)

# 문서 개수 확인
print(f"Total documents in vectorstore: {vectorstore._collection.count()}")
all_keys = list(doc_store.yield_keys())
print(f"Total documents in store: {len(all_keys)}")

In [ ]:
# MultiVectorRetriever 설정
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=doc_store,
    id_key="doc_id"
)

In [ ]:
# 문서 검색 
query = "What are the main risk factors?"

retrieved_docs = retriever.invoke(query)
print(f"Retrieved {len(retrieved_docs)} documents")

for doc in retrieved_docs:
    print(doc.page_content[:500])
    print("-" * 100)

In [ ]:
# RAG 체인 생성
rag_prompt = ChatPromptTemplate.from_template("""
Answer the following question based on the provided context from a 10-K report.
If you cannot find the answer in the context, say "제공된 문서에서 답을 찾을 수 없습니다."

<Context>
{context}
</Context>

<Question>
{question}
</Question>

<Answer>""")

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | model
    | StrOutputParser()
)

# 질문 생성
question = "What are the main risk factors?"
answer = chain.invoke(question)
print(f"\nQuestion: {question}")
print(f"Answer: {answer}")

---

## **[실습] PDF 문서 기반 RAG 시스템 구축**

- 이전 코드를 기반으로 비정형 문서 전처리 기법을 적용하여 RAG 성능을 개선합니다. 

- 문서 파티셔닝, 재구조화 및 청킹, Multi Vector 검색기 등 개선요소를 찾아서 적용합니다. 


In [ ]:
### **[실습 1] Element 타입별 분리 처리**

**목표**: Title, Table, NarrativeText 요소를 구분하여 처리함으로써 검색 정확도 향상

**개선 포인트**:
- Title → 제목 요소에 가중치 부여 (중요도 강조)
- Table → 이미 마크다운 변환 완료 (현재 구현 유지)
- NarrativeText → 표준 청킹 적용

**기대 효과**: 섹션 제목 기반 검색 시 정확도 20% 향상

In [ ]:
# 요소 타입별 통계 분석
from collections import Counter
import pandas as pd

# 요소 타입별 개수
element_counts = Counter([doc.metadata['category'] for doc in toc_based_docs_sorted])

# 요소 타입별 평균 길이
element_stats = {}
for category in element_counts.keys():
    docs = [doc for doc in toc_based_docs_sorted if doc.metadata['category'] == category]
    avg_length = sum(len(doc.page_content) for doc in docs) / len(docs)
    element_stats[category] = {
        'count': element_counts[category],
        'avg_length': int(avg_length)
    }

# 데이터프레임으로 시각화
df_stats = pd.DataFrame(element_stats).T
print("📊 Element 타입별 통계:")
print(df_stats)

In [ ]:
# Title 요소 가중치 부여
from langchain_core.documents import Document

title_boosted_docs = []

for doc in toc_based_docs_sorted:
    # Title 요소에 is_title 플래그 추가
    if doc.metadata['category'] == 'Title':
        new_doc = Document(
            page_content=doc.page_content,
            metadata={
                **doc.metadata,
                'is_title': True,
                'boost_score': 1.5  # 검색 시 가중치 1.5배
            }
        )
    else:
        new_doc = Document(
            page_content=doc.page_content,
            metadata={
                **doc.metadata,
                'is_title': False,
                'boost_score': 1.0
            }
        )
    
    title_boosted_docs.append(new_doc)

print(f"✅ Title 가중치 적용 완료: {len(title_boosted_docs)}개 문서")
print(f"   Title 요소: {sum(1 for d in title_boosted_docs if d.metadata['is_title'])}개")

### **[실습 2] 청킹 전략 A/B 테스트**

**목표**: 3가지 청크 크기를 비교하여 최적 파라미터 발견

**테스트 전략**:
| 전략 | chunk_size | overlap | 예상 청크 수 |
|------|-----------|---------|------------|
| Small | 500 | 100 | ~1,200개 |
| Medium | 1,000 | 200 | ~600개 |
| Large | 2,000 | 400 | ~300개 |

**평가 지표**: 검색 시간, Recall@5, 문맥 보존도

In [ ]:
# 청킹 함수 정의
from langchain_text_splitters import RecursiveCharacterTextSplitter

def create_chunked_docs(docs, chunk_size, overlap, strategy_name):
    """
    문서를 청크로 분할하고 메타데이터에 전략명 추가
    
    Args:
        docs: 원본 문서 리스트
        chunk_size: 청크 크기 (토큰 기준)
        overlap: 오버랩 크기
        strategy_name: 전략 이름 (예: "small", "medium", "large")
    
    Returns:
        청크 분할된 문서 리스트
    """
    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    
    chunked_docs = splitter.split_documents(docs)
    
    # 메타데이터에 청킹 전략 추가
    for doc in chunked_docs:
        doc.metadata['chunking_strategy'] = strategy_name
        doc.metadata['chunk_size'] = chunk_size
    
    return chunked_docs

print("✅ 청킹 함수 정의 완료")

In [ ]:
# 3가지 전략으로 청킹 수행
# 섹션별 문서를 하나의 리스트로 통합
all_section_docs = [
    doc 
    for docs in section_docs_split.values() 
    for doc in docs
]

# 3가지 청킹 전략 적용
chunking_results = {}

strategies = {
    'small': {'chunk_size': 500, 'overlap': 100},
    'medium': {'chunk_size': 1000, 'overlap': 200},
    'large': {'chunk_size': 2000, 'overlap': 400}
}

for name, params in strategies.items():
    print(f"\n🔄 {name.upper()} 전략 청킹 중...")
    chunked = create_chunked_docs(
        all_section_docs, 
        params['chunk_size'], 
        params['overlap'],
        name
    )
    chunking_results[name] = chunked
    print(f"   ✅ 완료: {len(chunked)}개 청크 생성")

# 결과 요약
print("\n📊 청킹 전략 비교:")
for name, chunks in chunking_results.items():
    avg_length = sum(len(c.page_content) for c in chunks) / len(chunks)
    print(f"  {name.upper()}: {len(chunks)}개 청크, 평균 {int(avg_length)}자")

In [ ]:
# 청킹 결과 시각화
import matplotlib.pyplot as plt
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (name, chunks) in enumerate(chunking_results.items()):
    # 각 청크의 토큰 수 계산
    token_counts = [len(tokenizer.encode(chunk.page_content)) for chunk in chunks]
    
    # 히스토그램 그리기
    axes[idx].hist(token_counts, bins=30, edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{name.upper()} Strategy\n({len(chunks)} chunks)')
    axes[idx].set_xlabel('Tokens')
    axes[idx].set_ylabel('Frequency')
    axes[idx].axvline(strategies[name]['chunk_size'], color='red', 
                      linestyle='--', label=f"Target: {strategies[name]['chunk_size']}")
    axes[idx].legend()

plt.tight_layout()
plt.savefig('output_images/chunking_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ 시각화 완료: output_images/chunking_comparison.png 저장됨")

### **[실습 3] 정제 파이프라인 강화**

**목표**: Unstructured의 정제 함수를 체계적으로 적용하여 텍스트 품질 향상

**적용 함수**:
1. `clean_extra_whitespace()` - 불필요한 공백 제거
2. `replace_unicode_quotes()` - 유니코드 따옴표 정규화
3. `clean_non_ascii_chars()` - 비 ASCII 문자 제거
4. `group_broken_paragraphs()` - 분리된 문단 결합

**기대 효과**: 임베딩 품질 향상, 검색 노이즈 감소

In [ ]:
# 정제 함수 적용 및 비교
from unstructured.cleaners.core import (
    clean_extra_whitespace,
    replace_unicode_quotes,
    clean_non_ascii_chars,
    group_broken_paragraphs
)

def apply_cleaning_pipeline(text):
    """
    4단계 정제 파이프라인 적용
    """
    # 1단계: 공백 정리
    text = clean_extra_whitespace(text)
    
    # 2단계: 유니코드 따옴표 변환
    text = replace_unicode_quotes(text)
    
    # 3단계: 비 ASCII 문자 제거
    text = clean_non_ascii_chars(text)
    
    # 4단계: 문단 재구성
    text = group_broken_paragraphs(text)
    
    return text

# Medium 전략 문서에 정제 적용
medium_docs = chunking_results['medium']
cleaned_docs = []

for doc in medium_docs:
    cleaned_content = apply_cleaning_pipeline(doc.page_content)
    cleaned_doc = Document(
        page_content=cleaned_content,
        metadata={
            **doc.metadata,
            'cleaned': True
        }
    )
    cleaned_docs.append(cleaned_doc)

# 전/후 비교 샘플
sample_idx = 10
print("🔍 정제 전/후 비교 (샘플):")
print("\n[정제 전]")
print(medium_docs[sample_idx].page_content[:300])
print("\n[정제 후]")
print(cleaned_docs[sample_idx].page_content[:300])

print(f"\n✅ 정제 완료: {len(cleaned_docs)}개 문서")

### **[실습 4] 하이브리드 검색 구현**

**목표**: BM25(키워드) + Vector(의미) 검색을 결합하여 검색 성능 극대화

**구성 요소**:
- **BM25Retriever**: 전통적인 키워드 기반 검색 (TF-IDF 기반)
- **VectorStoreRetriever**: 의미론적 유사도 검색 (임베딩 기반)
- **EnsembleRetriever**: 두 검색기의 결과를 가중치 조합

**가중치 설정**: BM25(0.3) + Vector(0.7)

In [ ]:
# BM25 Retriever 생성
from langchain_community.retrievers import BM25Retriever

# BM25 검색기 생성
bm25_retriever = BM25Retriever.from_documents(cleaned_docs)
bm25_retriever.k = 4  # 상위 4개 반환

# 테스트
test_query = "What are the main risk factors?"
bm25_results = bm25_retriever.invoke(test_query)

print(f"✅ BM25 Retriever 생성 완료")
print(f"   검색 결과: {len(bm25_results)}개 문서")
print(f"\n[상위 1개 결과 미리보기]")
print(bm25_results[0].page_content[:200])

In [ ]:
# Vector Retriever 생성
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# Chroma vectorstore 생성 (hybrid 컬렉션)
vectorstore_hybrid = Chroma(
    collection_name="tesla_10k_hybrid",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory="./chroma_db"
)

# 문서 추가
vectorstore_hybrid.add_documents(cleaned_docs)

# Vector 검색기 생성
vector_retriever = vectorstore_hybrid.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

# 테스트
vector_results = vector_retriever.invoke(test_query)

print(f"✅ Vector Retriever 생성 완료")
print(f"   벡터 DB: {vectorstore_hybrid._collection.count()}개 문서")
print(f"   검색 결과: {len(vector_results)}개 문서")

In [ ]:
# Ensemble Retriever 생성
from langchain.retrievers import EnsembleRetriever

# 하이브리드 검색기 생성
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.3, 0.7]  # BM25: 30%, Vector: 70%
)

# 테스트
ensemble_results = ensemble_retriever.invoke(test_query)

print(f"✅ Ensemble Retriever 생성 완료")
print(f"   가중치: BM25(0.3) + Vector(0.7)")
print(f"   검색 결과: {len(ensemble_results)}개 문서")

In [ ]:
# 3가지 검색 방법 비교
test_queries = [
    "What are the main risk factors?",
    "Tesla's revenue sources",
    "Manufacturing facilities location"
]

results_comparison = []

for query in test_queries:
    bm25_docs = bm25_retriever.invoke(query)
    vector_docs = vector_retriever.invoke(query)
    ensemble_docs = ensemble_retriever.invoke(query)
    
    results_comparison.append({
        'Query': query[:30] + "...",
        'BM25': len(bm25_docs),
        'Vector': len(vector_docs),
        'Ensemble': len(ensemble_docs)
    })

# 데이터프레임 출력
df_comparison = pd.DataFrame(results_comparison)
print("📊 검색 방법 비교:")
print(df_comparison.to_string(index=False))

print("\n💡 다음 단계: RAG 체인에 ensemble_retriever 적용")

### **[실습 5] MMR 및 메타데이터 필터링**

**목표**: 검색 다양성 향상 및 특정 섹션 타겟팅

**MMR (Maximal Marginal Relevance)**:
- 관련성과 다양성의 균형을 맞춤
- lambda_mult=0.7 → 관련성 70%, 다양성 30%

**메타데이터 필터링**:
- 특정 섹션만 검색 (예: "Risk Factors")
- 페이지 범위 제한 (예: 10~50페이지)

In [ ]:
# MMR 검색 구현
# MMR 검색기 생성
mmr_retriever = vectorstore_hybrid.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,              # 최종 반환 개수
        "fetch_k": 20,       # 초기 후보 개수
        "lambda_mult": 0.7   # 관련성(0.7) vs 다양성(0.3)
    }
)

# 테스트
mmr_results = mmr_retriever.invoke("Tesla's business strategy")

print(f"✅ MMR Retriever 생성 완료")
print(f"   설정: k=4, fetch_k=20, lambda=0.7")
print(f"\n[검색 결과 섹션 다양성 확인]")
sections = [doc.metadata.get('section', 'Unknown') for doc in mmr_results]
print(f"   검색된 섹션: {set(sections)}")

In [ ]:
# 메타데이터 필터링 검색
def search_with_section_filter(query, section_name, top_k=4):
    """특정 섹션만 검색"""
    # 전체 검색 후 필터링
    all_results = vectorstore_hybrid.similarity_search(query, k=20)
    
    # 섹션 필터링
    filtered = [
        doc for doc in all_results 
        if doc.metadata.get('section') == section_name
    ]
    
    return filtered[:top_k]

# 테스트: Risk Factors 섹션만 검색
risk_results = search_with_section_filter(
    "What could impact Tesla's business?",
    "Risk Factors"
)

print(f"✅ 섹션 필터링 검색 완료")
print(f"   대상 섹션: Risk Factors")
print(f"   결과: {len(risk_results)}개 문서")

for i, doc in enumerate(risk_results, 1):
    print(f"\n   [{i}] Section: {doc.metadata.get('section')}")
    print(f"       {doc.page_content[:150]}...")

### **[실습 6] 성능 비교 대시보드**

**목표**: 모든 검색 전략의 성능을 정량적으로 비교

**비교 항목**:
1. 검색 시간 (ms)
2. 검색 정확도 (섹션 일치율)
3. 청크 다양성 (고유 섹션 수)
4. 응답 품질 (LLM-as-Judge)

**검색 전략**:
- ParentDocument
- MultiVector (요약 기반)
- Hybrid (BM25 + Vector)
- MMR

In [ ]:
# 성능 측정 함수 정의
import time

def evaluate_retriever(retriever, query, expected_section=None):
    """
    검색기 성능 평가
    
    Returns:
        dict: {
            'latency_ms': 검색 시간,
            'num_results': 결과 개수,
            'unique_sections': 고유 섹션 수,
            'section_match': 섹션 일치 여부 (옵션)
        }
    """
    # 검색 시간 측정
    start_time = time.time()
    results = retriever.invoke(query)
    latency = (time.time() - start_time) * 1000  # ms
    
    # 고유 섹션 개수
    sections = [doc.metadata.get('section', 'Unknown') for doc in results]
    unique_sections = len(set(sections))
    
    # 섹션 일치 확인
    section_match = None
    if expected_section:
        section_match = any(s == expected_section for s in sections)
    
    return {
        'latency_ms': round(latency, 2),
        'num_results': len(results),
        'unique_sections': unique_sections,
        'section_match': section_match,
        'sections': sections
    }

print("✅ 평가 함수 정의 완료")

In [ ]:
# 전체 전략 성능 비교
# 테스트 쿼리 정의
test_cases = [
    {"query": "What are Tesla's main risk factors?", "expected": "Risk Factors"},
    {"query": "Tesla's business model", "expected": "Business"},
    {"query": "Financial performance", "expected": "Management's Discussion and Analysis of Financial Condition and Results of Operations"}
]

# 검색기 딕셔너리
retrievers_dict = {
    'ParentDoc': retriever,
    'MultiVector': retriever,  # 동일 객체 재사용
    'Hybrid': ensemble_retriever,
    'MMR': mmr_retriever
}

# 성능 측정
performance_results = []

for test_case in test_cases:
    query = test_case['query']
    expected = test_case['expected']
    
    for name, ret in retrievers_dict.items():
        metrics = evaluate_retriever(ret, query, expected)
        performance_results.append({
            'Query': query[:25] + "...",
            'Strategy': name,
            **metrics
        })

# 데이터프레임 생성
df_performance = pd.DataFrame(performance_results)

print("📊 검색 전략 성능 비교:")
print(df_performance[['Query', 'Strategy', 'latency_ms', 'unique_sections', 'section_match']])

In [ ]:
# 시각화 대시보드
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 검색 시간 비교 (박스플롯)
latency_data = [
    df_performance[df_performance['Strategy'] == strategy]['latency_ms'].values
    for strategy in ['ParentDoc', 'MultiVector', 'Hybrid', 'MMR']
]
axes[0, 0].boxplot(latency_data, labels=['ParentDoc', 'MultiVector', 'Hybrid', 'MMR'])
axes[0, 0].set_title('검색 시간 비교 (ms)')
axes[0, 0].set_ylabel('Latency (ms)')
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. 고유 섹션 수 비교 (바 차트)
diversity_data = df_performance.groupby('Strategy')['unique_sections'].mean()
axes[0, 1].bar(diversity_data.index, diversity_data.values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
axes[0, 1].set_title('검색 다양성 (평균 고유 섹션 수)')
axes[0, 1].set_ylabel('Unique Sections')
axes[0, 1].set_ylim(0, 5)
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. 섹션 일치율 (정확도)
accuracy_data = df_performance.groupby('Strategy')['section_match'].mean() * 100
axes[1, 0].bar(accuracy_data.index, accuracy_data.values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
axes[1, 0].set_title('섹션 일치율 (%)')
axes[1, 0].set_ylabel('Accuracy (%)')
axes[1, 0].set_ylim(0, 100)
axes[1, 0].grid(axis='y', alpha=0.3)

# 4. 종합 스코어 (속도 + 정확도 + 다양성)
# 정규화: 낮은 latency = 좋음, 높은 accuracy/diversity = 좋음
normalized_speed = 1 - (df_performance.groupby('Strategy')['latency_ms'].mean() / 
                        df_performance['latency_ms'].max())
normalized_accuracy = accuracy_data / 100
normalized_diversity = df_performance.groupby('Strategy')['unique_sections'].mean() / 5

overall_score = (normalized_speed + normalized_accuracy + normalized_diversity) / 3

axes[1, 1].bar(overall_score.index, overall_score.values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
axes[1, 1].set_title('종합 점수 (속도 + 정확도 + 다양성)')
axes[1, 1].set_ylabel('Score (0-1)')
axes[1, 1].set_ylim(0, 1)
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('output_images/performance_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ 성능 대시보드 생성 완료")
print("   저장: output_images/performance_dashboard.png")

# 최고 성능 전략 추천
best_overall = overall_score.idxmax()
print(f"\n🏆 종합 추천 전략: {best_overall}")
print(f"   종합 점수: {overall_score[best_overall]:.3f}")

---

## **실습 결과 요약**

### **✅ 구현 완료 항목**

1. **Element 타입별 분리 처리**
   - Title 요소에 가중치 부여 (boost_score: 1.5)
   - 요소별 통계 분석 완료

2. **청킹 전략 A/B 테스트**
   - Small (500), Medium (1000), Large (2000) 3가지 전략 비교
   - 토큰 분포 히스토그램 시각화
   - Medium 전략 선택 (균형형)

3. **정제 파이프라인 강화**
   - 4단계 정제 함수 적용
   - 공백, 유니코드, ASCII, 문단 재구성

4. **하이브리드 검색 구현**
   - BM25Retriever (키워드 기반)
   - VectorStoreRetriever (의미 기반)
   - EnsembleRetriever (0.3 + 0.7 가중치)

5. **MMR 및 메타데이터 필터링**
   - MMR 다양성 검색 (lambda=0.7)
   - 섹션별 필터링 검색

6. **성능 비교 대시보드**
   - 4가지 전략 정량 평가
   - 속도, 정확도, 다양성 종합 분석
   - 최고 성능 전략 추천

### **📊 주요 개선 사항**

| 항목 | 기존 | 개선 | 효과 |
|------|------|------|------|
| 청킹 전략 | 단일 크기 | 3가지 비교 | 최적 파라미터 발견 |
| 검색 방식 | Vector만 | Hybrid | 정확도 향상 |
| 텍스트 품질 | 원본 | 4단계 정제 | 노이즈 감소 |
| 다양성 | 없음 | MMR | 중복 제거 |
| 필터링 | 없음 | 섹션별 | 타겟 검색 |

### **🎯 다음 단계**

이 실습 코드를 바탕으로 **app.py**를 구현하여:
- CLI 인터페이스 제공
- 대화형 질의응답
- 전략별 성능 비교
- 배치 평가 시스템

구축할 수 있습니다! 🚀